# Phase 2 Analysis — Superstore Dataset

## Section 1: Discount Policy and Profit Erosion

**Objective:** Move from description to business interpretation. The core question is not *what* the discount levels are, but *whether they are justified* — and whether the profit losses observed are the result of discount policy or structural cost problems.

**Analytical frame:**
$$\text{Observed Margin} = \text{Base Margin}_{(d=0\%)} + \text{Discount-Driven Erosion}$$

Separating these two components determines whether a recommendation targets pricing, costs, or commercial policy.

**Questions addressed in this notebook:**
- **1.1** What is the optimal discount policy per sub-category to maximize profit?
- **1.2** Do discounts above 20% generate enough volume to compensate for margin loss?
- **1.3** What would happen to global profit if all discounts above 20% were eliminated?

---
## Setup

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
db_path = Path('../data/superstore.db')
conn = sqlite3.connect(db_path)
df = pd.read_sql_query('SELECT * FROM orders', conn)
conn.close()

print(f'Records loaded: {len(df):,}')
print(f'Columns: {list(df.columns)}')

---
## 1.1 — Optimal Discount Policy by Sub-Category

### Approach

Two-step analysis:

1. **Heatmap (visual support):** Profit margin across discount buckets for every sub-category. Shows where and how fast margins deteriorate as discounts increase — the visual basis for every recommendation.
2. **Breakeven calculation:** For each sub-category, identify the last discount bucket where margin remains positive. That upper bound becomes the recommended maximum discount.

> **Methodological note on elasticity:** This dataset contains historical transaction records, not a controlled pricing experiment. We cannot measure true price-demand elasticity. Volume patterns by discount bucket are treated as descriptive evidence, not causal claims.

In [ ]:
BUCKET_ORDER = ['0%', '1-10%', '11-20%', '21-30%', '31-50%', '>50%']

def assign_bucket(d):
    if d == 0:       return '0%'
    elif d <= 0.10:  return '1-10%'
    elif d <= 0.20:  return '11-20%'
    elif d <= 0.30:  return '21-30%'
    elif d <= 0.50:  return '31-50%'
    else:            return '>50%'

df['discount_bucket'] = df['Discount'].apply(assign_bucket)

# Weighted margin per sub-category / bucket: sum(Profit) / sum(Sales)
# Using sum/sum avoids distortion from small-value transactions.
bucket_stats = (
    df.groupby(['Sub-Category', 'discount_bucket'])
    .agg(profit=('Profit', 'sum'), sales=('Sales', 'sum'), n=('Profit', 'count'))
    .assign(margin=lambda x: x['profit'] / x['sales'])
    .reset_index()
)

pivot = (
    bucket_stats
    .pivot(index='Sub-Category', columns='discount_bucket', values='margin')
    .reindex(columns=BUCKET_ORDER)
)

# Sort by base margin (0% discount) descending — puts strongest sub-cats at top
pivot = pivot.sort_values('0%', ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

sns.heatmap(
    pivot * 100,
    annot=True,
    fmt='.1f',
    cmap='RdYlGn',
    center=0,
    vmin=-60,
    vmax=60,
    linewidths=0.4,
    linecolor='white',
    mask=pivot.isna(),
    cbar_kws={'label': 'Profit Margin (%)'},
    ax=ax
)

ax.set_title('Profit Margin (%) by Sub-Category and Discount Level', fontsize=13, pad=14)
ax.set_xlabel('Discount Bucket', fontsize=11)
ax.set_ylabel('')
ax.tick_params(axis='both', rotation=0)

plt.tight_layout()
plt.savefig('../visuals/09a_phase2_discount_margin_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### Breakeven Discount Calculation

For each sub-category, the breakeven discount is defined as the upper bound of the last bucket where margin remains non-negative. This becomes the ceiling for the recommended discount policy.

**Interpretation of edge cases:**
- If no data exists for intermediate buckets (e.g., 21-30%), the breakeven defaults to the last observed profitable bucket — a conservative estimate.
- If the sub-category is always profitable within observed discount levels, the recommendation is marked as `> observed range`.

In [ ]:
# Recommended max is the upper bound of the last profitable bucket,
# with one exception: if the last profitable bucket is exactly '0%',
# we cap at 10% (the gap to the next bucket is unknown, not necessarily zero).
REC_MAX_FROM_BUCKET = {
    '0%':    0.10,
    '1-10%': 0.10,
    '11-20%':0.20,
    '21-30%':0.30,
    '31-50%':0.40,
    '>50%':  None
}

def find_last_profitable_bucket(row):
    """Return the last bucket with non-negative margin before the first negative one."""
    last_ok = None
    for bucket in BUCKET_ORDER:
        m = row.get(bucket)
        if pd.isna(m):
            continue
        if m >= 0:
            last_ok = bucket
        else:
            return last_ok   # first negative bucket found — return last positive
    return last_ok           # never went negative in observed range

last_profitable_s = pivot.apply(find_last_profitable_bucket, axis=1)

In [ ]:
base_margin = pivot['0%'].rename('base_margin')

avg_discount = (
    df.groupby('Sub-Category')['Discount']
    .mean()
    .rename('avg_discount_applied')
)

overall_margin = (
    df.groupby('Sub-Category')
    .agg(profit=('Profit', 'sum'), sales=('Sales', 'sum'))
    .assign(m=lambda x: x['profit'] / x['sales'])['m']
    .rename('overall_margin')
)

last_profitable = last_profitable_s

def get_rec_max(bucket):
    if pd.isna(bucket):
        return None
    return REC_MAX_FROM_BUCKET.get(bucket)

rec_max = last_profitable.apply(get_rec_max).rename('rec_max')

def classify_status(row):
    om = row['Overall Margin']
    avg_d = row['Avg Discount Applied']
    rm = row['Recommended Max Disc']

    if om < 0:
        return 'Over-Discounted'
    if pd.isna(rm):
        return 'Healthy'    # always profitable in observed range
    if avg_d > rm:
        return 'At Risk' if om >= 0.05 else 'Over-Discounted'
    if avg_d > rm * 0.70:
        return 'At Risk'
    return 'Healthy'

rec = pd.DataFrame({
    'Base Margin (0% disc)': base_margin,
    'Avg Discount Applied':  avg_discount,
    'Overall Margin':        overall_margin,
    'Last Profitable Bucket':last_profitable,
    'Recommended Max Disc':  rec_max,
}).reset_index()

rec['Status'] = rec.apply(classify_status, axis=1)

# Sort: problem cases first, then At Risk, then Healthy — within each group by base margin
STATUS_ORDER = {'Over-Discounted': 0, 'At Risk': 1, 'Healthy': 2}
rec['_sort'] = rec['Status'].map(STATUS_ORDER)
rec = rec.sort_values(['_sort', 'Base Margin (0% disc)']).drop(columns='_sort')
rec.reset_index(drop=True, inplace=True)

In [ ]:
display_rec = rec.copy()

for col in ['Base Margin (0% disc)', 'Avg Discount Applied', 'Overall Margin']:
    display_rec[col] = display_rec[col].map(lambda x: f'{x:.1%}' if pd.notna(x) else '—')

display_rec['Recommended Max Disc'] = display_rec['Recommended Max Disc'].map(
    lambda x: f'{x:.0%}' if pd.notna(x) else '> observed range'
)

display_cols = [
    'Sub-Category', 'Base Margin (0% disc)', 'Avg Discount Applied',
    'Overall Margin', 'Last Profitable Bucket', 'Recommended Max Disc', 'Status'
]

STATUS_COLORS = {
    'Over-Discounted': '#f4cccc',
    'At Risk':         '#fce8b2',
    'Healthy':         '#d9ead3',
}

(
    display_rec[display_cols]
    .style
    .map(lambda v: f'background-color: {STATUS_COLORS.get(v, "")}', subset=['Status'])
    .set_caption('Table 1 — Discount Policy Recommendations by Sub-Category')
    .hide(axis='index')
)

### Key Findings — Section 1.1

**Critical reframe:** Every sub-category in the Superstore catalog is profitable at zero discount. Tables carries a 18.6% base margin; Bookcases 19.0%; even Supplies reaches 5.4%. There are no structural cost problems in the portfolio — the losses are entirely a function of discount policy.

**Over-Discounted sub-categories (overall margin negative):**

| Sub-Category | Base Margin | Avg Discount Applied | Overall Margin | Problem |
|---|---|---|---|---|
| Tables | 18.6% | 26.1% | −8.6% | Breakeven < 11%; avg discount is 2.4× the safe threshold |
| Bookcases | 19.0% | 21.1% | −3.0% | Avg discount marginally above the 20% ceiling |
| Supplies | 5.4% | 7.7% | −2.6% | Thin base margin; breakeven < 11%; any discount is high-risk |
| Machines | 38.2% | 30.6% | +1.8% | Avg discount at the breakeven threshold; margin nearly wiped |

**Notable pattern — Binders:** Base margin 48.0%, but 613 of 1,523 transactions carry discounts above 50% (−107% margin in that bucket). The average discount of 37.2% signals a structural over-discounting habit despite the category's strong base economics.

**Business recommendation:** The discount ceiling across the portfolio should be sub-category-specific, not a single blanket policy. A uniform 20% cap — common in retail — would be appropriate for most categories but is still too high for Tables (ceiling: 10%), Bookcases (ceiling: 20%), and Supplies (ceiling: 10%).